# Notebook 03 Part 2: `groupBy()` and Aggregations


## 1. Windows Local Setup Cell

Run this cell before creating `SparkSession`.

This cell helps PySpark use the correct Python environment and localhost settings on Windows.


In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


## 2. Import SparkSession

`SparkSession` is the starting point for working with PySpark.


In [2]:
from pyspark.sql import SparkSession

## 3. Create SparkSession

We will use the same Windows-safe SparkSession configuration.


In [3]:
spark = SparkSession.builder \
    .appName("pyspark") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 4. Create Sample DataFrame

We will create a small sales dataset.

This dataset has city, product, quantity, and amount columns.


In [4]:
data = [
    (1, "Amit", "Delhi", "Laptop", 1, 50000),
    (2, "Priya", "Mumbai", "Mobile", 2, 50000),
    (3, "Rahul", "Delhi", "Laptop", 1, 52000),
    (4, "Sneha", "Pune", "Tablet", 3, 54000),
    (5, "Neha", "Mumbai", "Mobile", 1, 25000),
    (6, "Karan", "Delhi", "Camera", 2, 60000),
    (7, "Ravi", "Pune", "Laptop", 1, 51000),
    (8, "Anjali", "Mumbai", "Tablet", 2, 36000),
    (9, "Vikas", "Delhi", "Mobile", 3, 75000),
    (10, "Meena", "Pune", "Camera", 1, 30000)
]

columns = ["order_id", "customer_name", "city", "product", "quantity", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

+--------+-------------+------+-------+--------+------+
|order_id|customer_name|  city|product|quantity|amount|
+--------+-------------+------+-------+--------+------+
|       1|         Amit| Delhi| Laptop|       1| 50000|
|       2|        Priya|Mumbai| Mobile|       2| 50000|
|       3|        Rahul| Delhi| Laptop|       1| 52000|
|       4|        Sneha|  Pune| Tablet|       3| 54000|
|       5|         Neha|Mumbai| Mobile|       1| 25000|
|       6|        Karan| Delhi| Camera|       2| 60000|
|       7|         Ravi|  Pune| Laptop|       1| 51000|
|       8|       Anjali|Mumbai| Tablet|       2| 36000|
|       9|        Vikas| Delhi| Mobile|       3| 75000|
|      10|        Meena|  Pune| Camera|       1| 30000|
+--------+-------------+------+-------+--------+------+



## 5. What is Aggregation?

Aggregation means summarizing data.

Example questions:

```text
1. How many orders are there in each city?
2. What is the total sales amount by city?
3. What is the average sales amount by product?
4. What is the minimum and maximum order amount?
```

In PySpark, we commonly use `groupBy()` with aggregation functions.


## 6. `groupBy()` Command

`groupBy()` is used to make groups based on one or more columns.

Syntax:

```python
df.groupBy("column_name")
```

Simple meaning:

```text
groupBy() = make groups
```

But `groupBy()` alone does not show output. We need an aggregation function after it.


## 7. Count Orders by City

Question:

```text
How many orders are there in each city?
```

For this, we use:

```python
df.groupBy("city").count()
```


In [ ]:
df.groupBy("city").count().show()

## 8. Count Orders by Product

Question:

```text
How many orders are there for each product?
```


In [5]:
df.groupBy("product").count().show()

+-------+-----+
|product|count|
+-------+-----+
| Laptop|    3|
| Camera|    2|
| Mobile|    3|
| Tablet|    2|
+-------+-----+



## 9. Import Aggregation Functions

For `sum()`, `avg()`, `min()`, and `max()`, we import functions from `pyspark.sql.functions`.

Important: With **agg** we use aliases like `spark_sum` and `spark_avg` because Python already has built-in functions named `sum`, `min`, and `max`.



In [12]:
from pyspark.sql.functions import sum as spark_sum, avg as spark_avg, min as spark_min, max as spark_max, count as spark_count

## 10. Total Sales Amount by City

Question:

```text
What is the total sales amount in each city?
```

Here we group by `city` and calculate sum of `amount`.


In [6]:
df.groupBy("city").sum("amount").show()

+------+-----------+
|  city|sum(amount)|
+------+-----------+
|Mumbai|     111000|
|  Pune|     135000|
| Delhi|     237000|
+------+-----------+



## 11. Total Quantity by Product

Question:

```text
How many total quantities were sold for each product?
```


In [7]:
df.groupBy("product").sum("quantity").show()

+-------+-------------+
|product|sum(quantity)|
+-------+-------------+
| Laptop|            3|
| Camera|            3|
| Mobile|            6|
| Tablet|            5|
+-------+-------------+



## 12. Average Amount by City

Question:

```text
What is the average order amount in each city?
```


In [8]:
df.groupBy("city").avg("amount").show()

+------+-----------+
|  city|avg(amount)|
+------+-----------+
|Mumbai|    37000.0|
|  Pune|    45000.0|
| Delhi|    59250.0|
+------+-----------+



## 13. Minimum Amount by City

Question:

```text
What is the smallest order amount in each city?
```


In [9]:
df.groupBy("city").min("amount").show()

+------+-----------+
|  city|min(amount)|
+------+-----------+
|Mumbai|      25000|
|  Pune|      30000|
| Delhi|      50000|
+------+-----------+



## 14. Maximum Amount by City

Question:

```text
What is the highest order amount in each city?
```


In [10]:
df.groupBy("city").max("amount").show()

+------+-----------+
|  city|max(amount)|
+------+-----------+
|Mumbai|      50000|
|  Pune|      54000|
| Delhi|      75000|
+------+-----------+



## 15. Why Use `agg()`?

`agg()` is used when we want to apply multiple aggregation functions together.

Example:

```text
For each city, calculate:
1. Total sales
2. Average sales
3. Minimum sales
4. Maximum sales
```


In [13]:
df.groupBy("city").agg(
    spark_sum("amount"),
    spark_avg("amount"),
    spark_min("amount"),
    spark_max("amount")
).show()

+------+-----------+-----------+-----------+-----------+
|  city|sum(amount)|avg(amount)|min(amount)|max(amount)|
+------+-----------+-----------+-----------+-----------+
|Mumbai|     111000|    37000.0|      25000|      50000|
|  Pune|     135000|    45000.0|      30000|      54000|
| Delhi|     237000|    59250.0|      50000|      75000|
+------+-----------+-----------+-----------+-----------+



## 16. Rename Aggregated Columns using `alias()`

Default aggregated column names can look like this:

```text
sum(amount)
avg(amount)
min(amount)
max(amount)
```

To make them cleaner, we use `alias()`.


In [14]:
df.groupBy("city").agg(
    spark_sum("amount").alias("total_sales"),
    spark_avg("amount").alias("average_sales"),
    spark_min("amount").alias("minimum_sales"),
    spark_max("amount").alias("maximum_sales")
).show()

+------+-----------+-------------+-------------+-------------+
|  city|total_sales|average_sales|minimum_sales|maximum_sales|
+------+-----------+-------------+-------------+-------------+
|Mumbai|     111000|      37000.0|        25000|        50000|
|  Pune|     135000|      45000.0|        30000|        54000|
| Delhi|     237000|      59250.0|        50000|        75000|
+------+-----------+-------------+-------------+-------------+



## 17. Count Using `agg()`

We can also use `count()` inside `agg()`.

Question:

```text
For each city, find number of orders and total sales.
```


In [15]:
df.groupBy("city").agg(
    spark_count("order_id").alias("total_orders"),
    spark_sum("amount").alias("total_sales")
).show()

+------+------------+-----------+
|  city|total_orders|total_sales|
+------+------------+-----------+
|Mumbai|           3|     111000|
|  Pune|           3|     135000|
| Delhi|           4|     237000|
+------+------------+-----------+



## 18. Group By Multiple Columns

We can group data using more than one column.

Example:

```text
Find total sales by city and product.
```


In [16]:
df.groupBy("city", "product").agg(
    spark_sum("amount").alias("total_sales"),
    spark_sum("quantity").alias("total_quantity")
).show()

+------+-------+-----------+--------------+
|  city|product|total_sales|total_quantity|
+------+-------+-----------+--------------+
| Delhi| Camera|      60000|             2|
|Mumbai| Tablet|      36000|             2|
|  Pune| Laptop|      51000|             1|
|  Pune| Camera|      30000|             1|
| Delhi| Laptop|     102000|             2|
|  Pune| Tablet|      54000|             3|
| Delhi| Mobile|      75000|             3|
|Mumbai| Mobile|      75000|             3|
+------+-------+-----------+--------------+



## 19. Sort Aggregated Result

After aggregation, we can sort the result.

Example:

```text
Show cities from highest sales to lowest sales.
```


In [17]:
from pyspark.sql.functions import col

df.groupBy("city").agg(
    spark_sum("amount").alias("total_sales")
).orderBy(col("total_sales").desc()).show()

+------+-----------+
|  city|total_sales|
+------+-----------+
| Delhi|     237000|
|  Pune|     135000|
|Mumbai|     111000|
+------+-----------+



## 20. Very Important Syntax Pattern

Most aggregation code follows this pattern:

```python
df.groupBy("group_column").agg(
    aggregation_function("numeric_column").alias("new_column_name")
)
```

Example:

```python
df.groupBy("city").agg(
    spark_sum("amount").alias("total_sales")
)
```


## 21. Simple Comparison

| Code | Meaning |
|---|---|
| `groupBy("city").count()` | Count rows for each city |
| `groupBy("city").sum("amount")` | Total amount for each city |
| `groupBy("city").avg("amount")` | Average amount for each city |
| `groupBy("city").min("amount")` | Minimum amount for each city |
| `groupBy("city").max("amount")` | Maximum amount for each city |
| `groupBy("city").agg(...)` | Apply multiple aggregations together |


##  Stop SparkSession

At the end of practice, stop SparkSession to release resources.


In [18]:
spark.stop()